# Lab 4: Sentiment analysis with Naive Bayes

Guidelines: https://github.com/MohammedMosuily/NLP_Lab4

Run cells from top to bottom. Dataset paths are relative to this folder.

## Tasks 1–6: Amazon unlocked-phone reviews
Ratings 1–2 are negative, 3 is neutral, and 4–5 are positive. The five preprocessing steps are lowercasing, punctuation/number removal, tokenization, stopword removal, and lemmatization. Keep negation words because they matter for sentiment. Source: https://www.kaggle.com/datasets/PromptCloudHQ/amazon-reviews-unlocked-mobile-phones

In [1]:
from pathlib import Path
import re
import kagglehub
import nltk
import pandas as pd
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
path = Path(kagglehub.dataset_download("PromptCloudHQ/amazon-reviews-unlocked-mobile-phones"))
df = pd.read_csv(path / "Amazon_Unlocked_Mobile.csv")
df = df.dropna(subset=["Reviews", "Rating"])
df = df.drop_duplicates(subset=["Reviews"])
df["sentiment"] = df["Rating"].map({1: "negative", 2: "negative", 3: "neutral", 4: "positive", 5: "positive"})
print(df["sentiment"].value_counts())

sentiment
positive    103773
negative     44351
neutral      14366
Name: count, dtype: int64


In [2]:
stop_words = set(stopwords.words("english")) - {"no", "nor", "not"}
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    tokens = text.split()
    tokens = [word for word in tokens if word not in stop_words]
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    return " ".join(tokens)

df["clean_review"] = df["Reviews"].apply(preprocess)
df = df[df["clean_review"].str.strip().ne("")]
X_train, X_test, y_train, y_test = train_test_split(
    df["clean_review"], df["sentiment"], test_size=0.2, random_state=42, stratify=df["sentiment"]
)
tfidf = TfidfVectorizer(max_features=1000)
train_vectors = tfidf.fit_transform(X_train)
test_vectors = tfidf.transform(X_test)
model = MultinomialNB()
model.fit(train_vectors, y_train)
predictions = model.predict(test_vectors)
print("Accuracy:", accuracy_score(y_test, predictions))
print(classification_report(y_test, predictions, zero_division=0))
labels = ["negative", "neutral", "positive"]
print(pd.DataFrame(confusion_matrix(y_test, predictions, labels=labels),
                   index=labels, columns=labels))

Accuracy: 0.8031413612565445
              precision    recall  f1-score   support

    negative       0.80      0.68      0.73      8866
     neutral       0.38      0.01      0.02      2872
    positive       0.81      0.97      0.88     20732

    accuracy                           0.80     32470
   macro avg       0.66      0.55      0.54     32470
weighted avg       0.77      0.80      0.76     32470

          negative  neutral  positive
negative      6024       24      2818
neutral        818       30      2024
positive       684       24     20024
